<a href="https://colab.research.google.com/github/Aravindr017/Natural_Language_Processing-Spam_Classification/blob/main/Model/BagofWords/NLP_BagofWords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Libraries

In [72]:
import nltk
import pandas as pd
import numpy as np
import string # because we are handling messages and we have to convert to lower case

# for stemming
from nltk.stem import PorterStemmer
# for lemmatization
from nltk.stem import WordNetLemmatizer

# for bag of words
from sklearn.feature_extraction.text import CountVectorizer

# for train test split
from sklearn.model_selection import train_test_split

# for label encoding
from sklearn.preprocessing import LabelEncoder

# for Logistic regression model - classification
from sklearn.linear_model import LogisticRegression

# for evaluation metrices
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

# for ensemble (boosting model)
from sklearn.ensemble import AdaBoostClassifier

# Read Data

In [4]:
file_path = '/content/drive/MyDrive/ICT - Ai Ml/Natural Language Processing/Data/spam.xlsx'
df_spam = pd.read_excel(file_path)
df_spam.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# EDA

In [5]:
# relevent data is present in first two columns only
df_spam = df_spam[['v1', 'v2']]
df_spam.head(3)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


In [6]:
df_spam['v1'].value_counts()

,count
v1,
ham,4825
spam,747


# Preprocessing - on sample text

## Punctuations Removal using user defined function

In [7]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [8]:
sample_text = "Hello! + Can we =test #1 ($50.99) _item? Yes, @John_Doe said: 'It's a -well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;"
sample_text

"Hello! + Can we =test #1 ($50.99) _item? Yes, @John_Doe said: 'It's a -well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;"

In [9]:
# function to remove punctations from string

def remove_punctuation(text):
  punctuationless_text = ''.join([i for i in text if i not in string.punctuation])
  # comparing each character in 'text' against the punctuation list.
  return punctuationless_text

In [10]:
print('before')
print('--------------------')
print(sample_text)
print('\n\n')
print('After')
print('--------------------')
punctuation_free_text = remove_punctuation(sample_text)
print(remove_punctuation(sample_text))

before
--------------------
Hello! + Can we =test #1 ($50.99) _item? Yes, @John_Doe said: 'It's a -well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;



After
--------------------
Hello  Can we test 1 5099 item Yes JohnDoe said Its a wellknown stateoftheart 50  100 match id 789—or is it Check examplecom file datacsv status active  pending ok all done


## Lowercasing

In [11]:

print('before')
print('--------------------')
print(punctuation_free_text)
print('\n\n')
print('After')
print('--------------------')
lowercase_text = punctuation_free_text.lower()  # convert the string to lower case
print(lowercase_text)

before
--------------------
Hello  Can we test 1 5099 item Yes JohnDoe said Its a wellknown stateoftheart 50  100 match id 789—or is it Check examplecom file datacsv status active  pending ok all done



After
--------------------
hello  can we test 1 5099 item yes johndoe said its a wellknown stateoftheart 50  100 match id 789—or is it check examplecom file datacsv status active  pending ok all done


## Tokenization

In [12]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [13]:
# we are converting the entire text into a list of unique words
# function to get tokens from the sentance
# make sure the i/p text should be punctuation removed and in lower case.
def tokenization(text):
  words_list = nltk.word_tokenize(text)
  return words_list

In [14]:
tokens = tokenization(lowercase_text)
tokens

['hello',
 'can',
 'we',
 'test',
 '1',
 '5099',
 'item',
 'yes',
 'johndoe',
 'said',
 'its',
 'a',
 'wellknown',
 'stateoftheart',
 '50',
 '100',
 'match',
 'id',
 '789—or',
 'is',
 'it',
 'check',
 'examplecom',
 'file',
 'datacsv',
 'status',
 'active',
 'pending',
 'ok',
 'all',
 'done']

## Stop Words Removal

In [15]:
nltk.download('stopwords')    # library for stopwords
stop_words_list = nltk.corpus.stopwords.words('english')
stop_words_list

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [16]:
# function to remove the stop words in the sentence after tokentization
def remove_stopwords(tokens):
  stop_words = nltk.corpus.stopwords.words('english')
  tokens_without_stopwords = [i for i in tokens if i not in stop_words]
  return tokens_without_stopwords

In [17]:
clean_tokens = remove_stopwords(tokens)

print('before')
print('--------------------')
print(tokens)
print(f'There are {len(tokens)} tokens in this sentence')
print('\n\n')
print('After')
print('--------------------')
print(clean_tokens)
print(f'There are {len(clean_tokens)} tokens in this sentence')


before
--------------------
['hello', 'can', 'we', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'its', 'a', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'is', 'it', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'all', 'done']
There are 31 tokens in this sentence



After
--------------------
['hello', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'done']
There are 24 tokens in this sentence


## Stemming

- Removing the prefix and sufix and only consider the root word. <br>
eg : going -> go

In [18]:
stem_obj = PorterStemmer()

def stemming(clean_tokens):
  stem_list = [stem_obj.stem(word) for word in clean_tokens]
  return stem_list



# stem_obj.stem(word) - will give the stem word for each word in the clean_tokens

In [19]:
stem_token_list = stemming(clean_tokens)

print('before')
print('--------------------')
print(clean_tokens)
print(f'There are {len(clean_tokens)} tokens in this sentence')
print('\n\n')
print('After')
print('--------------------')
print(stem_token_list)
print(f'There are {len(stem_token_list)} tokens in this sentence')

before
--------------------
['hello', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'done']
There are 24 tokens in this sentence



After
--------------------
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']
There are 24 tokens in this sentence


## Lemmatization

- Taking the root word with the meaning of the word <br>
eg : went -> go

In [20]:
nltk.download('wordnet')  # for lemmatization ( get words with meaning )

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [21]:
lemma_obj = WordNetLemmatizer()

def lemmatization(token):
  lemma_list = [lemma_obj.lemmatize(i) for i in token]
  return lemma_list

In [22]:
lemma_list = lemmatization(stem_token_list)

print('before')
print('--------------------')
print(stem_token_list)
print('\n\n')
print('After')
print('--------------------')
print(lemma_list)

before
--------------------
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']



After
--------------------
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']


# Preprocessing - on Dataset

##  Convert to lower case  

In [23]:
df_spam['v2'].head(10)

,v2
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
5,FreeMsg Hey there darling it's been 3 week's n...
6,Even my brother is not like to speak with me. ...
7,As per your request 'Melle Melle (Oru Minnamin...
8,WINNER!! As a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...


In [24]:
df_spam['v2'] = df_spam['v2'].astype('string')
# as a numerical value is present in the column so we have to convert the number to string to do the operations.

In [25]:
df_spam['lower_case'] = df_spam['v2'].str.lower()

In [26]:
df_spam['lower_case'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 5572 entries, 0 to 5571
Series name: lower_case
Non-Null Count  Dtype 
--------------  ----- 
5572 non-null   string
dtypes: string(1)
memory usage: 43.7 KB


## Punctuation Removal

In [27]:
df_spam['punctuation'] = df_spam['lower_case'].apply(remove_punctuation)

In [28]:
df_spam['punctuation'] = df_spam['punctuation'].apply(lambda x : x.replace('�', ''))

In [29]:
df_spam['punctuation'].head(10)

,punctuation
0,go until jurong point crazy available only in ...
1,ok lar joking wif u oni
2,free entry in 2 a wkly comp to win fa cup fina...
3,u dun say so early hor u c already then say
4,nah i dont think he goes to usf he lives aroun...
5,freemsg hey there darling its been 3 weeks now...
6,even my brother is not like to speak with me t...
7,as per your request melle melle oru minnaminun...
8,winner as a valued network customer you have b...
9,had your mobile 11 months or more u r entitled...


## Tokenization

In [30]:
df_spam['tokens'] = df_spam['punctuation'].apply(tokenization)

In [31]:
df_spam['tokens'].head(10)

,tokens
0,"[go, until, jurong, point, crazy, available, o..."
1,"[ok, lar, joking, wif, u, oni]"
2,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,"[u, dun, say, so, early, hor, u, c, already, t..."
4,"[nah, i, dont, think, he, goes, to, usf, he, l..."
5,"[freemsg, hey, there, darling, its, been, 3, w..."
6,"[even, my, brother, is, not, like, to, speak, ..."
7,"[as, per, your, request, melle, melle, oru, mi..."
8,"[winner, as, a, valued, network, customer, you..."
9,"[had, your, mobile, 11, months, or, more, u, r..."


## Stop Words Removal

In [32]:
df_spam['stop_words'] = df_spam['tokens'].apply(remove_stopwords)

In [33]:
df_spam['stop_words'].head(10)

,stop_words
0,"[go, jurong, point, crazy, available, bugis, n..."
1,"[ok, lar, joking, wif, u, oni]"
2,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, early, hor, u, c, already, say]"
4,"[nah, dont, think, goes, usf, lives, around, t..."
5,"[freemsg, hey, darling, 3, weeks, word, back, ..."
6,"[even, brother, like, speak, treat, like, aids..."
7,"[per, request, melle, melle, oru, minnaminungi..."
8,"[winner, valued, network, customer, selected, ..."
9,"[mobile, 11, months, u, r, entitled, update, l..."


## Stemming

In [34]:
df_spam['stem_words'] = df_spam['stop_words'].apply(stemming)

In [35]:
df_spam['stem_words'].head(10)

,stem_words
0,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joke, wif, u, oni]"
2,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,"[nah, dont, think, goe, usf, live, around, tho..."
5,"[freemsg, hey, darl, 3, week, word, back, id, ..."
6,"[even, brother, like, speak, treat, like, aid,..."
7,"[per, request, mell, mell, oru, minnaminungint..."
8,"[winner, valu, network, custom, select, receiv..."
9,"[mobil, 11, month, u, r, entitl, updat, latest..."


## Lemmatization

In [36]:
df_spam['lemma_words'] = df_spam['stem_words'].apply(lemmatization)

In [37]:
df_spam['lemma_words'].head(10)

,lemma_words
0,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joke, wif, u, oni]"
2,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,"[nah, dont, think, goe, usf, live, around, tho..."
5,"[freemsg, hey, darl, 3, week, word, back, id, ..."
6,"[even, brother, like, speak, treat, like, aid,..."
7,"[per, request, mell, mell, oru, minnaminungint..."
8,"[winner, valu, network, custom, select, receiv..."
9,"[mobil, 11, month, u, r, entitl, updat, latest..."


## Word Count

In [38]:
df_spam['lemma_word_count'] = df_spam['lemma_words'].apply(lambda x : len(x))
df_spam['initial_word_count'] = df_spam['punctuation'].apply(lambda x : len(x))

## DataFrame visualization

In [39]:
df_spam

,v1,v2,lower_case,punctuation,tokens,stop_words,stem_words,lemma_words,lemma_word_count,initial_word_count
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre...",16,102
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]",6,23
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...",23,149
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t...","[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[u, dun, say, earli, hor, u, c, alreadi, say]",9,43
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l...","[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[nah, dont, think, goe, usf, live, around, tho...",8,59
...,...,...,...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,this is the 2nd time we have tried 2 contact u...,this is the 2nd time we have tried 2 contact u...,"[this, is, the, 2nd, time, we, have, tried, 2,...","[2nd, time, tried, 2, contact, u, u, 750, poun...","[2nd, time, tri, 2, contact, u, u, 750, pound,...","[2nd, time, tri, 2, contact, u, u, 750, pound,...",20,151
5568,ham,Will �_ b going to esplanade fr home?,will �_ b going to esplanade fr home?,will b going to esplanade fr home,"[will, b, going, to, esplanade, fr, home]","[b, going, esplanade, fr, home]","[b, go, esplanad, fr, home]","[b, go, esplanad, fr, home]",5,34
5569,ham,"Pity, * was in mood for that. So...any other s...","pity, * was in mood for that. so...any other s...",pity was in mood for that soany other suggest...,"[pity, was, in, mood, for, that, soany, other,...","[pity, mood, soany, suggestions]","[piti, mood, soani, suggest]","[piti, mood, soani, suggest]",4,50
5570,ham,The guy did some bitching but I acted like i'd...,the guy did some bitching but i acted like i'd...,the guy did some bitching but i acted like id ...,"[the, guy, did, some, bitching, but, i, acted,...","[guy, bitching, acted, like, id, interested, b...","[guy, bitch, act, like, id, interest, buy, som...","[guy, bitch, act, like, id, interest, buy, som...",14,124


# Bag of Words

In [42]:
count_vectorizer_obj = CountVectorizer()

df_spam['clean_text'] = df_spam['lemma_words'].apply(lambda x : ' '.join(x))
df_spam.head(3)

,v1,v2,lower_case,punctuation,tokens,stop_words,stem_words,lemma_words,lemma_word_count,initial_word_count,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre...",16,102,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]",6,23,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...",23,149,free entri 2 wkli comp win fa cup final tkt 21...


In [48]:
count_vec = count_vectorizer_obj.fit_transform(df_spam['clean_text'])     # transforming the corpus to vector
count_vec.toarray()   # the vector representation to arrray

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [49]:
count_vec.shape   # return the dimension of the array

(5572, 7959)

# Model Building and Evaluation
- Logistic Regression

In [50]:
df_spam.head()

,v1,v2,lower_case,punctuation,tokens,stop_words,stem_words,lemma_words,lemma_word_count,initial_word_count,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ...",go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre...",16,102,go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]",6,23,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...",23,149,free entri 2 wkli comp win fa cup final tkt 21...
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t...","[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[u, dun, say, earli, hor, u, c, alreadi, say]",9,43,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l...","[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[nah, dont, think, goe, usf, live, around, tho...",8,59,nah dont think goe usf live around though


In [51]:
df_spam['v1'].unique()

array(['ham', 'spam'], dtype=object)

## Target and Features

In [52]:
X = count_vec
y = df_spam['v1']

## Train test split

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

## Encoding the target

In [58]:
label_encoder_obj = LabelEncoder()
y_train = label_encoder_obj.fit_transform(y_train)
y_test = label_encoder_obj.transform(y_test)

## Log Model - Classification

In [59]:
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

## Model Metrices

In [67]:
print(f'Accuracy of log model \t:  {(accuracy_score(y_test, y_pred)*100):.2f}%')
print(f'F1 Score of log model \t:  {(f1_score(y_test, y_pred)*100):.2f}%')
print(f'Recall of log model \t:  {(recall_score(y_test, y_pred)*100)}%')
print(f"Precision of log model \t: {precision_score(y_test, y_pred) * 100:.2f}%")


Accuracy of log model 	:  97.85%
F1 Score of log model 	:  91.30%
Recall of log model 	:  84.0%
Precision of log model 	: 100.00%


In [71]:
# # Inferance :
# # - As the dataset is unbalanced where only 13%  are spam messages and the rest are non spam so we have to make the model better:
#       - one way is to make the non spam message count equal to the spam message count
#       - else we have to use the boosting model where it focus on the unbalanced dataset and make model more accurate.

## Boosting Model

- As this is the unbalanced dataset we are going to use the boosting model

In [77]:
ada_boost_model = AdaBoostClassifier(
    estimator = LogisticRegression(),
    n_estimators = 20,
    learning_rate = 1
)

ada_boost_model.fit(X_train, y_train)
y_pred_boost = ada_boost_model.predict(X_test)


In [78]:
# Boosting Model Evaluation Metrices

print(f'Accuracy of Boosting model \t:  {(accuracy_score(y_test, y_pred_boost)*100):.2f}%')
print(f'F1 Score of Boosing model \t:  {(f1_score(y_test, y_pred_boost)*100):.2f}%')
print(f'Recall of Boosting model \t:  {(recall_score(y_test, y_pred_boost)*100)}%')
print(f"Precision of Boosting model \t: {precision_score(y_test, y_pred_boost) * 100:.2f}%")

Accuracy of Boosting model 	:  97.94%
F1 Score of Boosing model 	:  92.26%
Recall of Boosting model 	:  91.33333333333333%
Precision of Boosting model 	: 93.20%


In [79]:
# Inference:
# - Here we get better accuracy along with better recall (false negatives).